# 09 — Intelligence Dashboard

Generate a self-contained maritime intelligence dashboard from real AIS data.

The dashboard is a single HTML file with animated vessel tracks, gate-crossing
analytics, live counters, filters, and vessel detail cards — all powered by
deck.gl. No server required; open it in any browser.

This example downloads one day of NOAA data for the Port of Los Angeles,
derives tracks, and produces a full interactive dashboard.

In [ ]:
from neptune_ais import Neptune
from neptune_ais.viz import (
    DashboardConfig,
    InfrastructurePoint,
    generate_dashboard,
)
from neptune_ais.derive.crossings import GateLine

## 1. Download real AIS data

NOAA provides free archival AIS data for all US coastal waters (2009–present).
We download a single day for the LA / Long Beach port area.

In [ ]:
LA_BBOX = (-118.6, 33.3, -117.7, 34.1)  # (west, south, east, north)

n = Neptune(
    "2024-06-15",
    sources=["noaa"],
    bbox=LA_BBOX,
    cache_dir="/tmp/neptune_la_demo",
)
n.download()

In [ ]:
positions = n.positions().collect()
print(f"{len(positions):,} positions from {positions['mmsi'].n_unique():,} vessels")

## 2. Derive tracks

Segment positions into vessel tracks with WKB geometry and per-vertex
timestamps for animation.

In [ ]:
tracks = n.tracks(
    include_geometry=True,
    min_points=5,
    min_distance_m=500,
    refresh=True,
).collect()

print(f"{len(tracks)} tracks from {tracks['mmsi'].n_unique()} vessels")

## 3. Configure the dashboard

A `DashboardConfig` parameterizes the analysis:

- **`gate`** — a `GateLine` across the port entrance. Vessels crossing it are
  classified as inbound or outbound, feeding the live counter and time-series chart.
- **`infrastructure`** — named map markers for ports, anchorages, etc.
- **`speed`** — animation speed in seconds of vessel time per second of playback
  (7200 = 2 hours/second).

In [ ]:
gate = GateLine("LA Port Entrance", (33.70, -118.28), (33.70, -118.17))

infra = [
    InfrastructurePoint("Port of Los Angeles", 33.74, -118.27, "port"),
    InfrastructurePoint("Port of Long Beach", 33.75, -118.19, "port"),
    InfrastructurePoint("Angel Gate", 33.71, -118.25, "anchorage"),
]

config = DashboardConfig(
    title="PORT OF LOS ANGELES",
    description=(
        "Real NOAA AIS data for June 15, 2024 — vessel traffic in the "
        "LA/Long Beach port complex, the busiest container port in the "
        "Western Hemisphere."
    ),
    gate=gate,
    date_from="2024-06-15",
    date_to="2024-06-15",
    infrastructure=infra,
    speed=7200,
    zoom=11,
    center_lat=33.72,
    center_lon=-118.22,
)

## 4. Generate the dashboard

`generate_dashboard()` pre-computes all analytics (crossing counts, reversals,
daily time-series, vessel index) in Python, embeds them as JSON, and produces a
self-contained HTML file with deck.gl layers, filters, and playback controls.

`max_tracks=500` keeps the file size manageable (~5 MB).

In [ ]:
vessels_df = n.vessels().collect()

output = generate_dashboard(
    tracks,
    vessels=vessels_df if len(vessels_df) > 0 else None,
    config=config,
    output="dashboard.html",
    max_tracks=500,
)

import os
print(f"Dashboard: {output}")
print(f"File size: {os.path.getsize(output) / 1024:.0f} KB")

Open `dashboard.html` in any browser. The dashboard includes:

- **Animated vessel trails** with directional arrow markers
- **Gate crossing counter** — inbound/outbound split, same-hull reversals
- **Time-series chart** — daily crossing volume
- **Filters** — by flag, ship type, transit status, MMSI search
- **Layer toggles** — Trips, Heads, Tracks (static paths), Gates, Density, Infrastructure
- **Vessel detail card** — click any arrow marker to see vessel info
- **Keyboard shortcuts** — Space (play/pause), arrows (seek), 1/2/3 (speed)

## 5. Adapting for other ports

Change the bounding box, gate line, and infrastructure to analyze any US port:

In [ ]:
# Example: Port of Houston
#
# HOUSTON_BBOX = (-95.1, 29.3, -94.5, 29.8)
# gate = GateLine("Houston Ship Channel", (29.35, -94.77), (29.38, -94.72))
#
# n = Neptune("2024-06-15", sources=["noaa"], bbox=HOUSTON_BBOX)
# n.download()
# tracks = n.tracks(include_geometry=True).collect()
#
# config = DashboardConfig(
#     title="PORT OF HOUSTON",
#     description="Vessel traffic through the Houston Ship Channel.",
#     gate=gate,
#     date_from="2024-06-15",
#     date_to="2024-06-15",
#     speed=7200,
# )
# generate_dashboard(tracks, config=config, output="houston_dashboard.html")